## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import time

np.random.seed(42)
tf.random.set_seed(42)

print("=== ONNX Export (mit tf2onnx oder Simulation) ===\n")

# ── 1. Modell trainieren ──────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,), name='dense_1'),
    layers.Dense(64,  activation='relu', name='dense_2'),
    layers.Dense(32,  activation='relu', name='dense_3'),
    layers.Dense(10,  activation='softmax', name='output')
], name='mnist_onnx_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
_, acc = model.evaluate(x_test, y_test, verbose=0)
model.save('model_for_onnx.keras')
print(f"\nModell Test-Accuracy: {acc:.4f}")

# ── 2. Versuche tf2onnx ───────────────────────────────────
ONNX_AVAILABLE = False
try:
    import tf2onnx
    import onnx
    ONNX_AVAILABLE = True
    print("\ntf2onnx und onnx gefunden! Exportiere als ONNX...")
except ImportError:
    print("\ntf2onnx/onnx nicht verfügbar. Demonstriere Workflow mit detaillierten Print-Statements.")
    print("Installation: pip install tf2onnx onnx onnxruntime\n")

if ONNX_AVAILABLE:
    # ── Echter ONNX Export ────────────────────────────────
    # Methode 1: Über SavedModel
    saved_model_path = 'saved_for_onnx/'
    tf.saved_model.save(model, saved_model_path)

    spec = (tf.TensorSpec((None, 784), tf.float32, name='input'),)
    output_path = 'model.onnx'

    model_proto, external_tensor_storage = tf2onnx.convert.from_keras(
        model, input_signature=spec, output_path=output_path
    )

    onnx_size = os.path.getsize(output_path) / 1024
    keras_size = os.path.getsize('model_for_onnx.keras') / 1024
    print(f"\nONNX Export erfolgreich: {output_path}")
    print(f"  ONNX Dateigröße: {onnx_size:.1f} KB")
    print(f"  Keras Dateigröße: {keras_size:.1f} KB")

    # ONNX-Modell laden und Grafstruktur anzeigen
    onnx_model = onnx.load(output_path)
    print(f"\nONNX Modell-Informationen:")
    print(f"  IR Version: {onnx_model.ir_version}")
    print(f"  Opset Version: {onnx_model.opset_import[0].version}")
    print(f"  Knoten: {len(onnx_model.graph.node)}")
    print(f"  Eingaben: {[i.name for i in onnx_model.graph.input]}")
    print(f"  Ausgaben: {[o.name for o in onnx_model.graph.output]}")
    print("\nOperatoren im Graph:")
    for node in onnx_model.graph.node:
        print(f"  [{node.op_type}] {node.name}: {list(node.input)} → {list(node.output)}")

    # ONNX Inferenz (mit onnxruntime)
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(output_path)
        input_name = sess.get_inputs()[0].name
        onnx_preds = sess.run(None, {input_name: x_test[:5].astype(np.float32)})[0]
        keras_preds = model.predict(x_test[:5], verbose=0)
        print(f"\nONNX Inferenz (5 Samples): {np.argmax(onnx_preds, axis=1)}")
        print(f"Keras Inferenz (5 Samples): {np.argmax(keras_preds, axis=1)}")
        print(f"Identisch: {np.allclose(onnx_preds, keras_preds, atol=1e-5)}")
    except ImportError:
        print("onnxruntime nicht installiert für Inferenz.")

else:
    # ── Simulation: ONNX Workflow ohne echte Library ──────
    print("=== ONNX Export Workflow (Simulation) ===\n")

    def simulate_onnx_export(keras_model):
        """
        Simuliert den ONNX Export-Prozess.
        Zeigt detailliert, was tf2onnx intern tut.
        """
        print("Schritt 1: Keras-Modell → TensorFlow SavedModel")
        print("  tf.saved_model.save(model, 'saved_model/')")
        print("  → Speichert Graphstruktur + Gewichte als Protobuf\n")

        print("Schritt 2: TF Graphen analysieren")
        model_config = keras_model.get_config()
        print(f"  Modell-Name: {model_config.get('name', 'unknown')}")
        print(f"  Schichten: {len(model_config.get('layers', []))}")

        print("\nSchritt 3: Operatoren kartieren (TF → ONNX)")
        tf_to_onnx_ops = {
            'MatMul':   'Gemm',
            'BiasAdd':  'Add',
            'Relu':     'Relu',
            'Softmax':  'Softmax',
            'Reshape':  'Reshape',
        }
        print("  TF-Op       → ONNX-Op")
        for tf_op, onnx_op in tf_to_onnx_ops.items():
            print(f"  {tf_op:12s} → {onnx_op}")

        print("\nSchritt 4: ONNX Graph-Metadaten extrahieren")
        # Aus dem Keras-Modell
        graph_info = {
            'ir_version': 8,
            'opset_version': 17,
            'nodes': [],
            'inputs': [{'name': 'input', 'shape': ['batch', 784], 'dtype': 'float32'}],
            'outputs': [{'name': 'output', 'shape': ['batch', 10], 'dtype': 'float32'}]
        }

        for layer in keras_model.layers:
            if hasattr(layer, 'units'):
                graph_info['nodes'].append({
                    'op': 'Gemm',
                    'name': layer.name,
                    'inputs': ['input', f'{layer.name}/kernel', f'{layer.name}/bias'],
                    'outputs': [f'{layer.name}/output'],
                    'attributes': {'transB': 1}
                })
                if layer.activation.__name__ != 'linear':
                    act_name = layer.activation.__name__.capitalize()
                    graph_info['nodes'].append({
                        'op': act_name,
                        'name': f'{layer.name}/{act_name}',
                        'inputs': [f'{layer.name}/output'],
                        'outputs': [f'{layer.name}/{act_name}/output']
                    })

        print(f"\n  ONNX Graph-Struktur:")
        print(f"  Eingabe:  {graph_info['inputs'][0]}")
        print(f"  Ausgabe:  {graph_info['outputs'][0]}")
        print(f"  Knoten:   {len(graph_info['nodes'])}")
        print(f"\n  ONNX Knoten:")
        for node in graph_info['nodes']:
            print(f"  [{node['op']:10s}] {node['name']}")

        print("\nSchritt 5: ONNX IR serialisieren")
        print("  onnx_model.SerializeToString() → bytes")
        print("  with open('model.onnx', 'wb') as f: f.write(bytes)")
        print("  → model.onnx erstellt!")

        # Schätze ONNX-Größe
        total_params = sum(np.prod(w.shape) for w in keras_model.get_weights())
        estimated_size_kb = (total_params * 4) / 1024  # 4 Bytes pro float32
        print(f"\nGeschätzte ONNX-Größe: {estimated_size_kb:.1f} KB")
        print(f"  (basierend auf {total_params:,} float32 Parametern)")

        return graph_info

    # Simulation ausführen
    graph_info = simulate_onnx_export(model)

    # Metadaten als JSON speichern
    onnx_metadata = {
        'export_tool':    'tf2onnx (simuliert)',
        'keras_model':    model.name,
        'input_shape':    [None, 784],
        'output_shape':   [None, 10],
        'layers':         [{'name': l.name, 'type': type(l).__name__,
                            'params': l.count_params()}
                           for l in model.layers],
        'total_params':   model.count_params(),
        'opset_version':  17,
        'onnx_ops_used':  ['Gemm', 'Relu', 'Softmax'],
        'note':           'Simulierter Export - echte ONNX-Datei mit: pip install tf2onnx onnx'
    }

    with open('onnx_export_metadata.json', 'w') as f:
        json.dump(onnx_metadata, f, indent=2)
    print(f"\nMetadaten gespeichert: onnx_export_metadata.json")

    # Grafische Darstellung der Modellstruktur
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    ax.set_title('ONNX Modell-Graph (Simulation)', fontsize=14)

    # Zeichne Modell als Flowchart
    layer_info = [(l.name, type(l).__name__, l.count_params()) for l in model.layers]
    n = len(layer_info)
    y_positions = np.linspace(0.9, 0.1, n)

    for i, (name, ltype, params) in enumerate(layer_info):
        y = y_positions[i]
        rect = plt.Rectangle((0.1, y - 0.04), 0.8, 0.07,
                               fill=True, facecolor='lightblue', edgecolor='navy', linewidth=2)
        ax.add_patch(rect)
        ax.text(0.5, y, f"{name} [{ltype}]\n{params:,} Parameter",
                 ha='center', va='center', fontsize=9, fontweight='bold')
        if i < n - 1:
            ax.annotate('', xy=(0.5, y_positions[i+1] + 0.04),
                         xytext=(0.5, y - 0.04),
                         arrowprops=dict(arrowstyle='->', color='black', lw=2))

    # Keras → ONNX Annotation
    ax.text(0.95, 0.5, 'tf2onnx\n→\n.onnx', ha='center', va='center',
             fontsize=12, style='italic', color='darkblue',
             bbox=dict(boxstyle='round', facecolor='lightyellow'))

    plt.tight_layout()
    plt.savefig('E14_1_onnx_modellgraph.png', dpi=100)
    plt.show()

print("\nONNX Export abgeschlossen!")
print("ONNX ermöglicht Deployment in: ONNX Runtime, OpenVINO, TensorRT, CoreML, etc.")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

np.random.seed(42)
tf.random.set_seed(42)

print("=== Vollständige Modell-Kompression ===")
print("Pipeline: Großes Modell → Pruning → Quantisierung → Knowledge Distillation\n")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# ── 2. Großes Ausgangsmodell ──────────────────────────────
def build_large_model():
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(784,), name='d1'),
        layers.Dense(256, activation='relu', name='d2'),
        layers.Dense(128, activation='relu', name='d3'),
        layers.Dense(64,  activation='relu', name='d4'),
        layers.Dense(10,  activation='softmax', name='output')
    ], name='large_model')
    return model

large_model = build_large_model()
large_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Trainiere großes Modell (512-256-128-64)...")
large_model.fit(x_train, y_train, epochs=10, batch_size=256, validation_split=0.1, verbose=0)
_, large_acc = large_model.evaluate(x_test, y_test, verbose=0)
large_model.save('large_model.keras')
large_size = os.path.getsize('large_model.keras') / 1024
large_params = large_model.count_params()
print(f"Großes Modell: Acc={large_acc:.4f}, Params={large_params:,}, Size={large_size:.1f}KB\n")

# ── 3. Schritt 1: Magnitude-Based Pruning ────────────────
print("Schritt 1: Magnitude-Based Pruning")

def magnitude_prune_model(model, sparsity=0.5):
    """
    Magnitude-Based Pruning: Setzt kleinste Gewichte auf 0.
    sparsity=0.5 → 50% der Gewichte werden auf 0 gesetzt.
    """
    pruned_model = keras.models.clone_model(model)
    pruned_model.set_weights(model.get_weights())

    total_zeroed = 0
    total_weights = 0

    for layer in pruned_model.layers:
        if hasattr(layer, 'kernel'):
            weights = layer.kernel.numpy()
            threshold = np.percentile(np.abs(weights), sparsity * 100)
            mask = np.abs(weights) > threshold
            pruned_weights = weights * mask
            layer.kernel.assign(pruned_weights)
            total_zeroed  += np.sum(~mask)
            total_weights += weights.size

    actual_sparsity = total_zeroed / total_weights
    print(f"  Tatsächliche Sparsity: {actual_sparsity:.2%}")
    print(f"  Genullte Gewichte: {total_zeroed:,} / {total_weights:,}")
    return pruned_model, actual_sparsity

pruned_model, actual_sparsity = magnitude_prune_model(large_model, sparsity=0.5)

# Nach Pruning noch einmal finetunen
pruned_model.compile(optimizer=keras.optimizers.Adam(0.0001),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
pruned_model.fit(x_train, y_train, epochs=3, batch_size=256, verbose=0)
_, pruned_acc = pruned_model.evaluate(x_test, y_test, verbose=0)
pruned_model.save('pruned_model.keras')
pruned_size = os.path.getsize('pruned_model.keras') / 1024
# Effektive Parameter (Nicht-Null)
effective_params = sum(np.sum(w != 0) for w in pruned_model.get_weights())
print(f"  Pruned Modell: Acc={pruned_acc:.4f}, Effektive Params={effective_params:,}, "
      f"Size={pruned_size:.1f}KB\n")

# ── 4. Schritt 2: Post-Training Quantisierung ─────────────
print("Schritt 2: Post-Training Quantisierung (TFLite Int8)")

def quantize_model(model, x_rep):
    """Konvertiert zu TFLite mit Int8-Quantisierung."""
    def rep_data():
        for s in x_rep[:200]:
            yield [s.reshape(1, -1).astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = rep_data
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.TFLITE_BUILTINS
    ]
    converter.inference_input_type  = tf.float32
    converter.inference_output_type = tf.float32
    return converter.convert()

def eval_tflite(tflite_bytes, x_data, y_data, n=200):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp = interp.get_input_details()
    out = interp.get_output_details()
    correct = 0
    for i in range(min(n, len(x_data))):
        interp.set_tensor(inp[0]['index'], x_data[i:i+1].astype(np.float32))
        interp.invoke()
        pred = np.argmax(interp.get_tensor(out[0]['index']))
        if pred == y_data[i]:
            correct += 1
    return correct / min(n, len(x_data))

# Pruned Modell quantisieren
quant_bytes = quantize_model(pruned_model, x_train)
with open('quantized_pruned.tflite', 'wb') as f:
    f.write(quant_bytes)
quant_size = os.path.getsize('quantized_pruned.tflite') / 1024
quant_acc  = eval_tflite(quant_bytes, x_test, y_test)
print(f"  Quantisiertes Modell (Int8): Acc={quant_acc:.4f}, Size={quant_size:.1f}KB\n")

# ── 5. Schritt 3: Knowledge Distillation ─────────────────
print("Schritt 3: Knowledge Distillation (Kleines Modell lernt vom Großen)")

class DistillationModel(keras.Model):
    """
    Knowledge Distillation: Student lernt von Teacher.
    Loss = alpha * student_loss + (1-alpha) * distillation_loss
    """
    def __init__(self, student, teacher, temperature=5.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.student     = student
        self.teacher     = teacher
        self.temperature = temperature
        self.alpha       = alpha
        self.student_loss_tracker = keras.metrics.Mean(name='student_loss')
        self.distill_loss_tracker = keras.metrics.Mean(name='distill_loss')
        self.total_loss_tracker   = keras.metrics.Mean(name='total_loss')
        self.accuracy_tracker     = keras.metrics.SparseCategoricalAccuracy(name='accuracy')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.student_loss_tracker,
                self.distill_loss_tracker, self.accuracy_tracker]

    def train_step(self, data):
        x, y = data
        # Teacher-Vorhersagen (kein Gradient!)
        teacher_logits = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            student_logits = self.student(x, training=True)

            # Standardverlust (Student vs. echte Labels)
            student_loss = keras.losses.sparse_categorical_crossentropy(
                y, student_logits, from_logits=False)
            student_loss = tf.reduce_mean(student_loss)

            # Destillationsverlust (Student vs. Teacher mit Temperatur)
            t = self.temperature
            teacher_soft = tf.nn.softmax(teacher_logits / t)
            student_soft = tf.nn.softmax(student_logits / t)
            distill_loss = keras.losses.kullback_leibler_divergence(
                teacher_soft, student_soft)
            distill_loss = tf.reduce_mean(distill_loss) * (t ** 2)

            total_loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss

        grads = tape.gradient(total_loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.student_loss_tracker.update_state(student_loss)
        self.distill_loss_tracker.update_state(distill_loss)
        self.accuracy_tracker.update_state(y, student_logits)
        return {m.name: m.result() for m in self.metrics}

# Kleines Student-Modell
student_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='student_model')

print(f"  Student Params: {student_model.count_params():,} "
      f"(vs. Teacher: {large_model.count_params():,})")

# Student ohne Distillation (Baseline)
student_baseline = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='student_baseline')
student_baseline.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy', metrics=['accuracy'])
student_baseline.fit(x_train, y_train, epochs=10, batch_size=256, verbose=0)
_, baseline_acc = student_baseline.evaluate(x_test, y_test, verbose=0)
print(f"  Student (ohne Distillation): Acc={baseline_acc:.4f}")

# Distillations-Training
distill = DistillationModel(student_model, large_model, temperature=5.0, alpha=0.5)
distill.compile(optimizer=keras.optimizers.Adam(0.001))
distill.fit(x_train, y_train, epochs=15, batch_size=256, verbose=0)
_, distill_acc = student_model.evaluate(x_test, y_test, verbose=0)
student_model.save('distilled_student.keras')
distill_size = os.path.getsize('distilled_student.keras') / 1024
print(f"  Student (mit Distillation): Acc={distill_acc:.4f}, Size={distill_size:.1f}KB\n")

# ── 6. Ergebnisübersicht ──────────────────────────────────
print("="*60)
print("KOMPRESSION PIPELINE – ZUSAMMENFASSUNG")
print("="*60)
pipeline_results = [
    {'name': 'Original (512-256-128-64)', 'acc': large_acc,
     'size': large_size, 'params': large_params},
    {'name': 'Pruned (50% Sparsity)',     'acc': pruned_acc,
     'size': pruned_size, 'params': effective_params},
    {'name': 'Quantisiert (Int8 TFLite)', 'acc': quant_acc,
     'size': quant_size, 'params': effective_params},
    {'name': 'Student Baseline (64-32)',  'acc': baseline_acc,
     'size': distill_size, 'params': student_model.count_params()},
    {'name': 'Distilled Student',         'acc': distill_acc,
     'size': distill_size, 'params': student_model.count_params()},
]

print(f"{'Modell':30} | {'Acc':6} | {'Size(KB)':8} | {'Kompr.':8}")
print("-"*60)
for r in pipeline_results:
    compr = f"{large_size/r['size']:.1f}x" if r['size'] > 0 else 'N/A'
    print(f"{r['name']:30} | {r['acc']:.4f} | {r['size']:8.1f} | {compr:8}")

# ── 7. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Modell-Kompression Pipeline', fontsize=14)

names  = [r['name'].split('(')[0].strip() for r in pipeline_results]
accs   = [r['acc']  for r in pipeline_results]
sizes  = [r['size'] for r in pipeline_results]
colors = ['red', 'orange', 'green', 'blue', 'purple']

ax = axes[0]
ax.bar(range(len(names)), accs, color=colors, alpha=0.8)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_title('Accuracy Vergleich')
ax.set_ylabel('Test-Accuracy'); ax.set_ylim(0.9, 1.01); ax.grid(True, axis='y')

ax2 = axes[1]
ax2.bar(range(len(names)), sizes, color=colors, alpha=0.8)
ax2.set_xticks(range(len(names))); ax2.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax2.set_title('Modellgröße (KB)')
ax2.set_ylabel('Größe (KB)'); ax2.grid(True, axis='y')

ax3 = axes[2]
compressions = [large_size / s for s in sizes]
scatter = ax3.scatter(sizes, accs, c=range(len(names)), cmap='RdYlGn', s=150, zorder=5)
for i, (r, name) in enumerate(zip(pipeline_results, names)):
    ax3.annotate(name, (r['size'], r['acc']), textcoords='offset points',
                  xytext=(5, 3), fontsize=7)
ax3.set_title('Größe vs. Accuracy (Trade-off)')
ax3.set_xlabel('Modellgröße (KB)'); ax3.set_ylabel('Accuracy'); ax3.grid(True)

plt.tight_layout()
plt.savefig('E14_2_kompressions_pipeline.png', dpi=100)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy import stats
import time

np.random.seed(42)
tf.random.set_seed(42)

print("=== A/B Testing Framework für zwei ML-Modelle ===\n")

# ── 1. Zwei Modelle trainieren ────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# Modell A: Größeres Modell, langsamer aber genauer
model_a = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='model_A')
model_a.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Trainiere Modell A (256-128)...")
model_a.fit(x_train, y_train, epochs=8, batch_size=256, verbose=0)

# Modell B: Kleineres Modell, schneller aber ggf. weniger genau
model_b = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='model_B')
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Trainiere Modell B (64-32)...")
model_b.fit(x_train, y_train, epochs=8, batch_size=256, verbose=0)

_, overall_acc_a = model_a.evaluate(x_test, y_test, verbose=0)
_, overall_acc_b = model_b.evaluate(x_test, y_test, verbose=0)
print(f"\nModell A (256-128): Test-Acc={overall_acc_a:.4f}")
print(f"Modell B (64-32):   Test-Acc={overall_acc_b:.4f}\n")

# ── 2. A/B Testing Framework ──────────────────────────────
class ABTestFramework:
    """
    A/B Testing Framework für ML-Modelle.
    Teilt Anfragen 50/50 auf, trackt Metriken.
    """
    def __init__(self, model_a, model_b, split=0.5):
        self.models  = {'A': model_a, 'B': model_b}
        self.split   = split  # Anteil für Modell A
        self.results = {
            'A': {'predictions': [], 'correct': [], 'confidences': [], 'latencies': []},
            'B': {'predictions': [], 'correct': [], 'confidences': [], 'latencies': []}
        }
        self.request_count = {'A': 0, 'B': 0}

    def route_request(self):
        """Entscheidet, welches Modell eine Anfrage bekommt."""
        return 'A' if np.random.rand() < self.split else 'B'

    def process_request(self, x, y_true):
        """
        Verarbeitet eine Anfrage.
        Gibt Antwort + Modell-Variante zurück.
        """
        variant = self.route_request()
        model   = self.models[variant]

        # Inferenz mit Latenzmessung
        t_start = time.perf_counter()
        probs   = model.predict(x.reshape(1, -1), verbose=0)[0]
        latency = (time.perf_counter() - t_start) * 1000  # ms

        prediction = int(np.argmax(probs))
        confidence = float(probs[prediction])
        correct    = int(prediction == y_true)

        self.results[variant]['predictions'].append(prediction)
        self.results[variant]['correct'].append(correct)
        self.results[variant]['confidences'].append(confidence)
        self.results[variant]['latencies'].append(latency)
        self.request_count[variant] += 1

        return {
            'variant': variant, 'prediction': prediction,
            'confidence': confidence, 'correct': correct,
            'latency_ms': latency
        }

    def get_metrics(self, variant):
        """Berechnet Metriken für eine Variante."""
        r = self.results[variant]
        if not r['correct']:
            return None
        return {
            'n_requests': len(r['correct']),
            'accuracy':   np.mean(r['correct']),
            'avg_confidence': np.mean(r['confidences']),
            'avg_latency_ms': np.mean(r['latencies']),
            'p95_latency_ms': np.percentile(r['latencies'], 95),
        }

    def statistical_test(self):
        """Führt t-Test auf Accuracy-Unterschied durch."""
        a_correct = self.results['A']['correct']
        b_correct = self.results['B']['correct']
        if not a_correct or not b_correct:
            return None
        t_stat, p_value = stats.ttest_ind(a_correct, b_correct)
        return {'t_statistic': t_stat, 'p_value': p_value,
                'significant': p_value < 0.05}

    def recommend_winner(self):
        """Empfiehlt den Gewinner basierend auf Accuracy + Latenz."""
        m_a = self.get_metrics('A')
        m_b = self.get_metrics('B')
        if m_a is None or m_b is None:
            return None

        # Gewichtete Score-Funktion (70% Accuracy + 30% Latenz-Vorteil)
        max_latency = max(m_a['avg_latency_ms'], m_b['avg_latency_ms'])
        score_a = 0.7 * m_a['accuracy'] + 0.3 * (1 - m_a['avg_latency_ms'] / max_latency)
        score_b = 0.7 * m_b['accuracy'] + 0.3 * (1 - m_b['avg_latency_ms'] / max_latency)
        winner  = 'A' if score_a > score_b else 'B'
        return {'score_A': score_a, 'score_B': score_b, 'winner': winner}

# ── 3. A/B Test durchführen ───────────────────────────────
N_REQUESTS = 1000
framework  = ABTestFramework(model_a, model_b, split=0.5)

print(f"Starte A/B Test mit {N_REQUESTS} simulierten Anfragen (50/50-Split)...")

# Shuffle Test-Daten für realistische Simulation
perm    = np.random.permutation(len(x_test))
x_sim   = x_test[perm[:N_REQUESTS]]
y_sim   = y_test[perm[:N_REQUESTS]]

for i in range(N_REQUESTS):
    framework.process_request(x_sim[i], int(y_sim[i]))
    if (i + 1) % 200 == 0:
        m_a = framework.get_metrics('A')
        m_b = framework.get_metrics('B')
        if m_a and m_b:
            print(f"  [{i+1:4d}] A: n={m_a['n_requests']}, Acc={m_a['accuracy']:.3f}, "
                  f"Latenz={m_a['avg_latency_ms']:.2f}ms | "
                  f"B: n={m_b['n_requests']}, Acc={m_b['accuracy']:.3f}, "
                  f"Latenz={m_b['avg_latency_ms']:.2f}ms")

# ── 4. Ergebnisse ─────────────────────────────────────────
metrics_a = framework.get_metrics('A')
metrics_b = framework.get_metrics('B')
stat_test  = framework.statistical_test()
winner_rec = framework.recommend_winner()

print(f"\n{'='*60}")
print("A/B TEST ERGEBNISSE")
print(f"{'='*60}")
for variant, m in [('A', metrics_a), ('B', metrics_b)]:
    print(f"\nModell {variant}:")
    for k, v in m.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\nStatistischer Test (t-Test):")
print(f"  t-Statistik: {stat_test['t_statistic']:.4f}")
print(f"  p-Wert:      {stat_test['p_value']:.6f}")
print(f"  Signifikant (p<0.05): {stat_test['significant']}")

print(f"\nGewinner-Empfehlung:")
print(f"  Score A: {winner_rec['score_A']:.4f}")
print(f"  Score B: {winner_rec['score_B']:.4f}")
print(f"  → Empfehlung: Modell {winner_rec['winner']}")

# ── 5. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('A/B Testing: Modell A vs. Modell B', fontsize=14)

# Kumulative Accuracy über Zeit
ax = axes[0, 0]
cum_acc_a = np.cumsum(framework.results['A']['correct']) / (np.arange(len(framework.results['A']['correct'])) + 1)
cum_acc_b = np.cumsum(framework.results['B']['correct']) / (np.arange(len(framework.results['B']['correct'])) + 1)
ax.plot(cum_acc_a, label=f'Modell A (final={metrics_a["accuracy"]:.3f})', color='blue')
ax.plot(cum_acc_b, label=f'Modell B (final={metrics_b["accuracy"]:.3f})', color='red')
ax.set_title('Kumulative Accuracy über Requests')
ax.set_xlabel('Anzahl Requests (pro Modell)'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(True)

# Konfidenz-Verteilung
ax2 = axes[0, 1]
ax2.hist(framework.results['A']['confidences'], bins=30, alpha=0.7, color='blue',
          label=f'A (mean={metrics_a["avg_confidence"]:.3f})', density=True)
ax2.hist(framework.results['B']['confidences'], bins=30, alpha=0.7, color='red',
          label=f'B (mean={metrics_b["avg_confidence"]:.3f})', density=True)
ax2.set_title('Konfidenz-Verteilung')
ax2.set_xlabel('Confidence'); ax2.set_ylabel('Dichte')
ax2.legend(); ax2.grid(True)

# Latenz-Boxplot
ax3 = axes[1, 0]
ax3.boxplot([framework.results['A']['latencies'],
              framework.results['B']['latencies']],
             labels=['Modell A', 'Modell B'])
ax3.set_title('Latenz-Verteilung (ms)')
ax3.set_ylabel('Latenz (ms)'); ax3.grid(True, axis='y')
for i, (v, m) in enumerate([(framework.results['A']['latencies'], metrics_a),
                               (framework.results['B']['latencies'], metrics_b)], 1):
    ax3.text(i, np.max(v)*0.95, f'P95: {m["p95_latency_ms"]:.1f}ms',
              ha='center', fontsize=8)

# Score-Vergleich
ax4 = axes[1, 1]
score_keys = ['accuracy', 'avg_confidence']
# Normalisiere für Radar-Chart
vals_a = [metrics_a['accuracy'], metrics_a['avg_confidence'],
           1 - metrics_a['avg_latency_ms'] / max(metrics_a['avg_latency_ms'], metrics_b['avg_latency_ms'])]
vals_b = [metrics_b['accuracy'], metrics_b['avg_confidence'],
           1 - metrics_b['avg_latency_ms'] / max(metrics_a['avg_latency_ms'], metrics_b['avg_latency_ms'])]
labels_sc = ['Accuracy', 'Konfidenz', 'Latenz-Score']
x_pos = np.arange(len(labels_sc))
ax4.bar(x_pos - 0.2, vals_a, 0.4, label='Modell A', color='blue', alpha=0.8)
ax4.bar(x_pos + 0.2, vals_b, 0.4, label='Modell B', color='red',  alpha=0.8)
ax4.set_xticks(x_pos); ax4.set_xticklabels(labels_sc)
ax4.set_title('Metrik-Vergleich (normalisiert)')
ax4.set_ylabel('Score'); ax4.legend(); ax4.grid(True, axis='y')

winner_label = f"Empfohlener Gewinner: Modell {winner_rec['winner']}"
ax4.set_xlabel(winner_label, fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('E14_3_ab_testing.png', dpi=100)
plt.show()

print("\nA/B Testing abgeschlossen!")
